# 煙霧偵測模型訓練 — Google Colab

**模型：** YOLOv8n（實驗驗證：438 張資料量下，增強反而降低 mAP50，直接訓練最佳）  
**基準：** mAP50 0.6665（第一次訓練結果）  
**目標：** 匯出 ONNX 放到 `models/smoke_detector.onnx` 啟用條件 3

**開始前：** 上方選單 → `執行階段 → 變更執行階段類型 → T4 GPU`

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. 安裝套件

In [ ]:
!pip install -q roboflow ultralytics onnx onnxsim

## 2. 下載資料集

填入你的 Roboflow API Key：[取得方式](https://app.roboflow.com/) → Settings → Roboflow API

In [ ]:
from roboflow import Roboflow

API_KEY = "YOUR_ROBOFLOW_API_KEY"  # ← 替換為你的 Key

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("naruesuan-university").project("smoke-gxoy3")
versions = project.versions()
print(f"可用版本：{[v.version for v in versions]}")
dataset = versions[-1].download("yolov8", location="/content/smoke_dataset")
print(f"✅ 下載完成：{dataset.location}")

## 3. 建立 data.yaml

In [ ]:
import yaml, glob, os, shutil, random

BASE = "/content/smoke_dataset"

def find_dir(base, candidates):
    for c in candidates:
        imgs = glob.glob(f"{base}/{c}/*.jpg") + glob.glob(f"{base}/{c}/*.png")
        if imgs:
            return c, len(imgs)
    return None, 0

train_rel, n_train = find_dir(BASE, ["train/images", "images/train"])
val_rel,   n_val   = find_dir(BASE, ["valid/images", "val/images", "images/val"])

if not val_rel:
    print(f"未找到驗證集，從 train ({n_train} 張) 切割 10%...")
    train_lbl = train_rel.replace("images", "labels")
    all_imgs  = sorted(glob.glob(f"{BASE}/{train_rel}/*.jpg") + glob.glob(f"{BASE}/{train_rel}/*.png"))
    random.seed(42); random.shuffle(all_imgs)
    split = max(5, int(len(all_imgs) * 0.1))
    for d in ["valid/images", "valid/labels"]:
        os.makedirs(f"{BASE}/{d}", exist_ok=True)
    for img_path in all_imgs[:split]:
        fname = os.path.basename(img_path)
        stem  = os.path.splitext(fname)[0]
        shutil.move(img_path, f"{BASE}/valid/images/{fname}")
        lbl = f"{BASE}/{train_lbl}/{stem}.txt"
        if os.path.exists(lbl):
            shutil.move(lbl, f"{BASE}/valid/labels/{stem}.txt")
    val_rel = "valid/images"
    n_train = len(glob.glob(f"{BASE}/{train_rel}/*.jpg") + glob.glob(f"{BASE}/{train_rel}/*.png"))
    n_val   = split

print(f"train={n_train} 張，val={n_val} 張")

DATA_YAML = f"{BASE}/data.yaml"
with open(DATA_YAML, "w") as f:
    yaml.dump({"path": BASE, "train": train_rel, "val": val_rel,
               "nc": 1, "names": ["smoke"]}, f, default_flow_style=False)
print("✅ data.yaml：")
!cat /content/smoke_dataset/data.yaml

## 4. 訓練模型（YOLOv8n）

T4 GPU 大約需要 **20–30 分鐘**

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print(f"裝置：{'GPU (' + torch.cuda.get_device_name(0) + ')' if device == 0 else 'CPU'}")

DATA_YAML = "/content/smoke_dataset/data.yaml"
model = YOLO("yolov8n.pt")
model.train(
    data=DATA_YAML,
    epochs=120,
    batch=16 if device == 0 else 4,
    imgsz=640,
    device=device,
    project="/content/runs",
    name="smoke_detector",
    patience=25,
    lr0=0.01, lrf=0.005,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0, mixup=0.05,
    save=True, save_period=10, exist_ok=True,
)

## 5. 驗證指標

In [ ]:
import glob as _glob
from ultralytics import YOLO
from IPython.display import Image as IPImage, display

DATA_YAML  = "/content/smoke_dataset/data.yaml"
candidates = sorted(_glob.glob("/content/runs/smoke_detector*/weights/best.pt"))
assert candidates, "❌ 找不到 best.pt"
best_pt = candidates[-1]
print(f"模型：{best_pt}\n")

metrics = YOLO(best_pt).val(data=DATA_YAML, verbose=False)
b = metrics.box
print("=" * 40)
print(f"  mAP50    : {b.map50:.4f}")
print(f"  mAP50-95 : {b.map:.4f}")
print(f"  Precision: {b.mp:.4f}")
print(f"  Recall   : {b.mr:.4f}")
print(f"  F1 Score : {2*b.mp*b.mr/(b.mp+b.mr+1e-9):.4f}")
print("=" * 40)

for p in sorted(_glob.glob("/content/runs/smoke_detector*/*.png")):
    display(IPImage(p))

## 6. 匯出 ONNX

In [ ]:
from ultralytics import YOLO
YOLO(best_pt).export(format="onnx", imgsz=640, simplify=True, opset=17)
onnx_path = best_pt.replace(".pt", ".onnx")
print(f"✅ ONNX：{onnx_path}")

## 7. 下載模型

In [ ]:
import shutil, os
from google.colab import files

dst = "/content/smoke_detector.onnx"
shutil.copy(onnx_path, dst)
print(f"大小：{os.path.getsize(dst)/1024/1024:.1f} MB")
files.download(dst)
print("\n下載後複製到 cgr_detection/models/smoke_detector.onnx")

## 8. （選用）下載 .pt 權重備份

In [ ]:
from google.colab import files
files.download(best_pt)